### Markov Chain Multi-Touch Attribution

This notebook performs multi-touch attribution using a first-order Markov chain with removal effect.

### High-level idea
1. Each marketing journey is a sequence of touchpoints like `alpha > beta > gamma`.
2. We add:
   - a starting state `"Start"` at the beginning
   - an absorbing state `"Conversion"` or `"Null"` (no conversion) at the end.
   So: `Start -> alpha -> beta -> gamma -> Conversion`

3. We estimate transition probabilities `P(next_state | current_state)` from all journeys.

4. We compute the probability that someone starting at `"Start"` eventually reaches `"Conversion"`. This uses absorbing Markov chain math:
   - transient states: Start + all channels
   - absorbing states: Conversion, Null
   - fundamental matrix: `N = (I - Q)^(-1)`
   - absorption probabilities: `B = N * R`
   - conversion probability = `B["Start", "Conversion"]`

5. **Removal effect attribution**:
   For each channel `c`:
   - Remove that channel from all paths.
   - Rebuild the Markov chain.
   - Recompute conversion probability.
   - The drop in conversion probability = that channel's contribution.

6. We normalize those contributions across all channels to get fractional credit.

We'll output:
- Attribution by channel (share of conversions)
- Estimated conversions credited (using total conversions in the data)
- Optional revenue-weighted version using `total_conversion_value`.

In [13]:
import os 
pwd = os.getcwd()
print("Current working directory:", pwd)

Current working directory: c:\Users\91957\Desktop\MS Admissions\Concordia University\MachineLearningProjects\RetailMarkovChain\retail-measurement-lab\notebooks


In [14]:
import pandas as pd
import numpy as np
from collections import defaultdict

data_path = "../etl/Data.csv"
df_raw = pd.read_csv(data_path, sep=";")

df_raw.head()

,path,total_conversions,total_conversion_value,total_null
0,eta > iota > alpha > eta,1,0.244,3
1,iota > iota > iota > iota,2,3.195,6
2,alpha > iota > alpha > alpha > alpha > iota > ...,2,6.754,6
3,beta > eta,1,2.402,3
4,iota > eta > theta > lambda > lambda > theta >...,0,0.000,2


### Step 1: Expand paths into individual journeys

For each row:
- If `path = "A > B"` and `total_conversions = 3`, `total_null = 2`, `total_conversion_value = 120`:
  - We create 3 journeys: ["Start","A","B","Conversion"] each worth value 120/3 = 40
  - We create 2 journeys: ["Start","A","B","Null"] each worth value 0

We'll build a list:
`journeys = [(path_list, did_convert_bool, value_float), ...]`

In [15]:
def explode_journeys(df):
    journeys = []

    has_value_col = "total_conversion_value" in df.columns

    for _, row in df.iterrows():
        # split "A > B > C" into ["A","B","C"]
        steps = [s.strip() for s in str(row["path"]).split(">")]
        steps = [s for s in steps if s]  # drop any empty strings

        conversions = int(row.get("total_conversions", 0))
        nulls = int(row.get("total_null", 0))

        # value per converting journey on this path
        if has_value_col and conversions > 0:
            avg_val = row["total_conversion_value"] / conversions
        else:
            avg_val = 0.0

        # add converting journeys
        for _ in range(conversions):
            journeys.append( (["Start"] + steps + ["Conversion"], True, avg_val) )

        # add non-converting journeys
        for _ in range(nulls):
            journeys.append( (["Start"] + steps + ["Null"], False, 0.0) )

    return journeys

journeys = explode_journeys(df_raw)
len(journeys), journeys[:3]

(88387,
 [(['Start', 'eta', 'iota', 'alpha', 'eta', 'Conversion'], True, 0.244),
  (['Start', 'eta', 'iota', 'alpha', 'eta', 'Null'], False, 0.0),
  (['Start', 'eta', 'iota', 'alpha', 'eta', 'Null'], False, 0.0)])

### Step 2: Build transition probability matrix

We count every adjacent pair in every journey:
- "Email" -> "Direct"
- "Direct" -> "Conversion"
etc.

Then:
P(next_state = Y | current_state = X)
  = count(X -> Y) / sum_over_all_Y count(X -> Y)

We'll output `P_base`, a square DataFrame of transition probabilities.

In [16]:
def build_transition_matrix(journeys):
    trans_counts = defaultdict(int)
    outgoing_counts = defaultdict(int)
    states = set()

    for path, _, _ in journeys:
        for i in range(len(path) - 1):
            a, b = path[i], path[i+1]
            trans_counts[(a, b)] += 1
            outgoing_counts[a] += 1
            states.add(a)
            states.add(b)

    states = sorted(states)

    P = pd.DataFrame(0.0, index=states, columns=states)
    for (a, b), c in trans_counts.items():
        P.loc[a, b] = c / outgoing_counts[a]

    return P

P_base = build_transition_matrix(journeys)
P_base.head(10)

,Conversion,Null,Start,alpha,beta,delta,epsilon,eta,gamma,iota,kappa,lambda,mi,theta,zeta
Conversion,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Null,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Start,0.000000,0.000000,0.0,0.326360,0.137113,0.000034,0.004922,0.160804,0.007874,0.231109,0.003451,0.043411,0.000091,0.083621,0.001211
alpha,0.052941,0.187677,0.0,0.645053,0.012315,0.000019,0.007132,0.014672,0.001235,0.038870,0.001736,0.013337,0.000000,0.021836,0.003178
beta,0.024111,0.082695,0.0,0.083232,0.342069,0.000073,0.009581,0.258009,0.002633,0.118802,0.003267,0.019235,0.000000,0.046297,0.009996
delta,0.119048,0.404762,0.0,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.071429,0.047619,0.190476,0.000000,0.000000,0.000000
epsilon,0.085384,0.272069,0.0,0.171089,0.044702,0.000000,0.102428,0.078148,0.006432,0.096639,0.010130,0.057083,0.000000,0.051777,0.024120
eta,0.096051,0.327432,0.0,0.106217,0.104027,0.000184,0.024687,0.196183,0.001729,0.071964,0.008137,0.018602,0.000000,0.036927,0.007860
gamma,0.051454,0.184564,0.0,0.203579,0.073826,0.000000,0.012864,0.078859,0.104027,0.150447,0.000000,0.059284,0.000000,0.071588,0.009508
iota,0.044051,0.157073,0.0,0.103582,0.068486,0.000039,0.015375,0.057667,0.003703,0.432946,0.007221,0.035976,0.000092,0.053229,0.020561


### Step 3: Compute baseline conversion probability

We treat:
- "Conversion" and "Null" as absorbing states
- every other state (including "Start") as transient

Absorbing chain math:
1. Reorder states so transients first, absorbing last.
2. Split the full transition matrix P into:
   - Q = transient -> transient
   - R = transient -> absorbing
3. Fundamental matrix: N = (I - Q)^(-1)
4. Absorption matrix: B = N @ R
5. Baseline conversion probability is B["Start","Conversion"]

We'll also capture:
- total observed conversions (to scale attribution later)
- total observed conversion value

In [17]:
def absorbing_conversion_probability(P,
                                     start_state="Start",
                                     conversion_state="Conversion",
                                     null_state="Null"):
    all_states = list(P.index)

    absorbing_states = [conversion_state, null_state]
    transient_states = [s for s in all_states if s not in absorbing_states]

    ordered_states = transient_states + absorbing_states
    P_ord = P.loc[ordered_states, ordered_states]

    t_len = len(transient_states)

    Q = P_ord.iloc[:t_len, :t_len].values  # transient -> transient
    R = P_ord.iloc[:t_len, t_len:].values  # transient -> absorbing

    I = np.eye(t_len)
    try:
        N = np.linalg.inv(I - Q)
    except np.linalg.LinAlgError:
        # pseudo-inverse fallback if singular
        N = np.linalg.pinv(I - Q)

    B = N @ R  # absorption probabilities

    start_idx = transient_states.index(start_state)
    conv_abs_idx = [conversion_state, null_state].index(conversion_state)

    conv_prob = B[start_idx, conv_abs_idx]

    return conv_prob, {
        "ordered_states": ordered_states,
        "transient_states": transient_states,
        "absorbing_states": [conversion_state, null_state],
        "Q": Q,
        "R": R,
        "N": N,
        "B": B
    }

base_conv_prob, debug_info = absorbing_conversion_probability(P_base)

total_conversions_observed = int(df_raw.get("total_conversions", 0).sum())
total_conversion_value_observed = float(df_raw.get("total_conversion_value", 0).sum())

base_conv_prob, total_conversions_observed, total_conversion_value_observed

(np.float64(0.22384513559686386), 19785, 74802.97158721084)

### Step 4: Removal effect

For each channel c:
1. Remove c from every journey.
2. Recompute the conversion probability.
3. The drop = base_conv_prob - conv_prob_without_c.
   This is the "lift" (a.k.a. contribution) of that channel.

We DO NOT remove:
- "Start"
- "Conversion"
- "Null"
because those are structural

In [18]:
def remove_channel_from_journeys(journeys, channel_to_remove):
    new_journeys = []
    for path, did_convert, val in journeys:
        # Drop that channel
        new_path = [s for s in path if s != channel_to_remove]

        # Collapse accidental duplicates like ["Start","Start","Email"]
        cleaned_path = []
        for node in new_path:
            if not cleaned_path or node != cleaned_path[-1]:
                cleaned_path.append(node)

        new_journeys.append((cleaned_path, did_convert, val))
    return new_journeys


def build_markov_and_prob(journeys):
    P = build_transition_matrix(journeys)
    conv_prob, _ = absorbing_conversion_probability(P)
    return conv_prob, P


all_states = list(P_base.index)
channels = [s for s in all_states if s not in ["Start", "Conversion", "Null"]]

channel_effects = {}

for ch in channels:
    journeys_minus_ch = remove_channel_from_journeys(journeys, ch)

    try:
        conv_prob_minus_ch, _ = build_markov_and_prob(journeys_minus_ch)
    except Exception:
        # if removing this channel destroys the path to Conversion, treat prob as 0
        conv_prob_minus_ch = 0.0

    lift = base_conv_prob - conv_prob_minus_ch

    channel_effects[ch] = {
        "conv_prob_without": conv_prob_minus_ch,
        "lift": lift
    }

channel_effects

{'alpha': {'conv_prob_without': np.float64(0.2238451355968638),
  'lift': np.float64(5.551115123125783e-17)},
 'beta': {'conv_prob_without': np.float64(0.2238451355968638),
  'lift': np.float64(5.551115123125783e-17)},
 'delta': {'conv_prob_without': np.float64(0.22384513559686375),
  'lift': np.float64(1.1102230246251565e-16)},
 'epsilon': {'conv_prob_without': np.float64(0.22384513559686375),
  'lift': np.float64(1.1102230246251565e-16)},
 'eta': {'conv_prob_without': np.float64(0.22384513559686384),
  'lift': np.float64(2.7755575615628914e-17)},
 'gamma': {'conv_prob_without': np.float64(0.22384513559686378),
  'lift': np.float64(8.326672684688674e-17)},
 'iota': {'conv_prob_without': np.float64(0.22384513559686375),
  'lift': np.float64(1.1102230246251565e-16)},
 'kappa': {'conv_prob_without': np.float64(0.22384513559686378),
  'lift': np.float64(8.326672684688674e-17)},
 'lambda': {'conv_prob_without': np.float64(0.2238451355968638),
  'lift': np.float64(5.551115123125783e-17)},
 

### Step 5: Attribution table

We convert each channel's lift into:
- `weight`: lift / sum(lift)  → that channel's share of influence
- `attributed_conversions`: weight × total_conversions_observed
- `attributed_value`: weight × total_conversion_value_observed

We then sort by attributed_conversions

In [19]:
attr_df = (
    pd.DataFrame
    .from_dict(channel_effects, orient="index")
    .rename_axis("channel")
    .reset_index()
)

total_lift = attr_df["lift"].sum()

# Define a tiny epsilon to detect "basically zero"
EPS = 1e-12

if abs(total_lift) < EPS:
    # No channel had meaningful incremental impact on conversion probability
    attr_df["weight"] = 0.0
else:
    attr_df["weight"] = attr_df["lift"] / total_lift

attr_df["attributed_conversions"] = attr_df["weight"] * total_conversions_observed
attr_df["attributed_value"] = attr_df["weight"] * total_conversion_value_observed

attr_df = attr_df.sort_values("attributed_conversions", ascending=False)
attr_df

,channel,conv_prob_without,lift,weight,attributed_conversions,attributed_value
0,alpha,0.223845,5.551115e-17,0.0,0.0,0.0
1,beta,0.223845,5.551115e-17,0.0,0.0,0.0
2,delta,0.223845,1.110223e-16,0.0,0.0,0.0
3,epsilon,0.223845,1.110223e-16,0.0,0.0,0.0
4,eta,0.223845,2.775558e-17,0.0,0.0,0.0
5,gamma,0.223845,8.326673e-17,0.0,0.0,0.0
6,iota,0.223845,1.110223e-16,0.0,0.0,0.0
7,kappa,0.223845,8.326673e-17,0.0,0.0,0.0
8,lambda,0.223845,5.551115e-17,0.0,0.0,0.0
9,mi,0.223845,8.326673e-17,0.0,0.0,0.0


### Step 6: Sanity check

We verify that:
- The sum of attributed conversions ~= total_conversions_observed
- The sum of attributed value ~= total_conversion_value_observed

Small floating-point differences are normal.

In [20]:
check_sum_conversions = attr_df["attributed_conversions"].sum()
check_sum_value = attr_df["attributed_value"].sum()

print("Observed total conversions:", total_conversions_observed)
print("Sum of attributed conversions:", check_sum_conversions)
print()
print("Observed total conversion value:", total_conversion_value_observed)
print("Sum of attributed value:", check_sum_value)

attr_df.reset_index(drop=True)

Observed total conversions: 19785
Sum of attributed conversions: 0.0

Observed total conversion value: 74802.97158721084
Sum of attributed value: 0.0


,channel,conv_prob_without,lift,weight,attributed_conversions,attributed_value
0,alpha,0.223845,5.551115e-17,0.0,0.0,0.0
1,beta,0.223845,5.551115e-17,0.0,0.0,0.0
2,delta,0.223845,1.110223e-16,0.0,0.0,0.0
3,epsilon,0.223845,1.110223e-16,0.0,0.0,0.0
4,eta,0.223845,2.775558e-17,0.0,0.0,0.0
5,gamma,0.223845,8.326673e-17,0.0,0.0,0.0
6,iota,0.223845,1.110223e-16,0.0,0.0,0.0
7,kappa,0.223845,8.326673e-17,0.0,0.0,0.0
8,lambda,0.223845,5.551115e-17,0.0,0.0,0.0
9,mi,0.223845,8.326673e-17,0.0,0.0,0.0


### Exporting attribution results

We'll export to `../etl/markov_attribution_results.csv`

In [21]:
output_path = "../etl/data/outputs/markov_attribution_results.csv"
attr_df.to_csv(output_path, index=False)
output_path

'../etl/data/outputs/markov_attribution_results.csv'